<a href="https://colab.research.google.com/github/zakieassad/pcb-defect-detection-yolo/blob/main/notebooks/04_colab_final_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [2]:
!git clone https://github.com/zakieassad/pcb-defect-detection-yolo.git
%cd pcb-defect-detection-yolo

Cloning into 'pcb-defect-detection-yolo'...
remote: Enumerating objects: 3101, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3101 (delta 0), reused 0 (delta 0), pack-reused 3099 (from 2)
Receiving objects: 100% (3101/3101), 44.55 MiB | 31.94 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/pcb-defect-detection-yolo


In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 59.8 MB/s eta 0:00:00


In [4]:
from ultralytics import YOLO
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.87
CUDA: True
GPU: Tesla T4


In [5]:
from pathlib import Path
import shutil

import cv2
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# RUTAS
# =========================

PROJECT_ROOT = Path.cwd()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Descargar DeepPCB
!wget -q https://github.com/tangsanli5201/DeepPCB/archive/refs/heads/master.zip -O data/raw/DeepPCB-master.zip
!unzip -q data/raw/DeepPCB-master.zip -d data/raw/

DEEPPCB_DIR = DATA_RAW_DIR / "DeepPCB-master"
PCB_DATA_DIR = DEEPPCB_DIR / "PCBData"

TRAINVAL_FILE = PCB_DATA_DIR / "trainval.txt"
TEST_FILE = PCB_DATA_DIR / "test.txt"

YOLO_DATASET_DIR = DATA_PROCESSED_DIR / "deep_pcb_yolo"

assert DEEPPCB_DIR.exists(), "No se encontró DeepPCB."
assert PCB_DATA_DIR.exists(), "No se encontró PCBData."
assert TRAINVAL_FILE.exists(), "No se encontró trainval.txt."
assert TEST_FILE.exists(), "No se encontró test.txt."

print("Dataset original descargado correctamente.")


# =========================
# FUNCIONES
# =========================

def read_split_file(file_path: Path) -> list[str]:
    with open(file_path, "r", encoding="utf-8") as file:
        return [line.strip() for line in file if line.strip()]


def resolve_test_image_path(image_rel_path: str) -> Path:
    logical_path = PCB_DATA_DIR / image_rel_path

    if logical_path.exists():
        return logical_path

    return logical_path.with_name(
        logical_path.stem + "_test" + logical_path.suffix
    )


def build_samples_dataframe(samples: list[str], original_split: str) -> pd.DataFrame:
    records = []

    for sample in samples:
        image_rel_path, annotation_rel_path = sample.split()

        records.append({
            "sample_id": Path(image_rel_path).stem,
            "group": Path(image_rel_path).parts[0],
            "original_split": original_split,
            "image_path": resolve_test_image_path(image_rel_path),
            "annotation_path": PCB_DATA_DIR / annotation_rel_path,
        })

    return pd.DataFrame(records)


def validate_original_bbox(
    x_min: int,
    y_min: int,
    x_max: int,
    y_max: int,
    image_width: int,
    image_height: int,
) -> None:
    assert 0 <= x_min < x_max <= image_width
    assert 0 <= y_min < y_max <= image_height


def convert_bbox_to_yolo(
    x_min: int,
    y_min: int,
    x_max: int,
    y_max: int,
    image_width: int,
    image_height: int,
) -> tuple[float, float, float, float]:
    box_width = x_max - x_min
    box_height = y_max - y_min

    x_center = x_min + box_width / 2
    y_center = y_min + box_height / 2

    return (
        x_center / image_width,
        y_center / image_height,
        box_width / image_width,
        box_height / image_height,
    )


def convert_annotation_file_to_yolo(annotation_path: Path, image_path: Path) -> list[str]:
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"No se pudo leer la imagen: {image_path}")

    image_height, image_width = image.shape[:2]
    yolo_lines = []

    with open(annotation_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            values = line.split()

            if len(values) != 5:
                raise ValueError(
                    f"Formato inválido en {annotation_path}, línea {line_number}: {line}"
                )

            x_min, y_min, x_max, y_max, original_class_id = map(int, values)

            if original_class_id not in range(1, 7):
                raise ValueError(
                    f"Clase inválida {original_class_id} en {annotation_path}, línea {line_number}"
                )

            validate_original_bbox(
                x_min=x_min,
                y_min=y_min,
                x_max=x_max,
                y_max=y_max,
                image_width=image_width,
                image_height=image_height,
            )

            x_center, y_center, box_width, box_height = convert_bbox_to_yolo(
                x_min=x_min,
                y_min=y_min,
                x_max=x_max,
                y_max=y_max,
                image_width=image_width,
                image_height=image_height,
            )

            yolo_class_id = original_class_id - 1

            yolo_lines.append(
                f"{yolo_class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{box_width:.6f} "
                f"{box_height:.6f}"
            )

    return yolo_lines


def reset_yolo_dataset_directory(output_dir: Path) -> None:
    if output_dir.exists():
        shutil.rmtree(output_dir)

    for split_name in ["train", "val", "test"]:
        (output_dir / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (output_dir / "labels" / split_name).mkdir(parents=True, exist_ok=True)


def process_split(split_df: pd.DataFrame, split_name: str, output_dir: Path) -> dict:
    images_output_dir = output_dir / "images" / split_name
    labels_output_dir = output_dir / "labels" / split_name

    processed_images = 0
    processed_labels = 0
    processed_annotations = 0

    for _, sample in split_df.iterrows():
        source_image_path = sample["image_path"]
        source_annotation_path = sample["annotation_path"]

        target_image_path = images_output_dir / f"{sample['sample_id']}{source_image_path.suffix}"
        target_label_path = labels_output_dir / f"{sample['sample_id']}.txt"

        shutil.copy2(source_image_path, target_image_path)

        yolo_annotations = convert_annotation_file_to_yolo(
            annotation_path=source_annotation_path,
            image_path=source_image_path,
        )

        with open(target_label_path, "w", encoding="utf-8") as file:
            file.write("\n".join(yolo_annotations))
            if yolo_annotations:
                file.write("\n")

        processed_images += 1
        processed_labels += 1
        processed_annotations += len(yolo_annotations)

    return {
        "split": split_name,
        "images": processed_images,
        "label_files": processed_labels,
        "annotations": processed_annotations,
    }


# =========================
# RECONSTRUCCIÓN DE SPLITS
# =========================

trainval_samples = read_split_file(TRAINVAL_FILE)
test_samples = read_split_file(TEST_FILE)

df_trainval = build_samples_dataframe(trainval_samples, original_split="trainval")
df_test = build_samples_dataframe(test_samples, original_split="test")

df_train, df_val = train_test_split(
    df_trainval,
    test_size=0.20,
    random_state=42,
    stratify=df_trainval["group"],
)

df_test_final = df_test.copy()

print(f"Train: {len(df_train)} imágenes")
print(f"Validation: {len(df_val)} imágenes")
print(f"Test: {len(df_test_final)} imágenes")


# =========================
# GENERACIÓN DATASET YOLO
# =========================

reset_yolo_dataset_directory(YOLO_DATASET_DIR)

processing_results = [
    process_split(df_train, "train", YOLO_DATASET_DIR),
    process_split(df_val, "val", YOLO_DATASET_DIR),
    process_split(df_test_final, "test", YOLO_DATASET_DIR),
]

df_processing_results = pd.DataFrame(processing_results)
display(df_processing_results)

assert df_processing_results["images"].sum() == 1500
assert df_processing_results["label_files"].sum() == 1500
assert df_processing_results["annotations"].sum() == 10013

print("Dataset YOLO generado correctamente.")


# =========================
# DATA.YAML
# =========================

data_yaml_content = f"""path: {YOLO_DATASET_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: open
  1: short
  2: mousebite
  3: spur
  4: copper
  5: pin-hole
"""

data_yaml_path = YOLO_DATASET_DIR / "data.yaml"

with open(data_yaml_path, "w", encoding="utf-8") as file:
    file.write(data_yaml_content)

print(data_yaml_content)
print(f"data.yaml creado en: {data_yaml_path}")

Dataset original descargado correctamente.
Train: 800 imágenes
Validation: 200 imágenes
Test: 500 imágenes


,split,images,label_files,annotations
0,train,800,800,5513
1,val,200,200,1360
2,test,500,500,3140


Dataset YOLO generado correctamente.
path: /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo
train: images/train
val: images/val
test: images/test

names:
  0: open
  1: short
  2: mousebite
  3: spur
  4: copper
  5: pin-hole

data.yaml creado en: /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/data.yaml


In [6]:
from ultralytics import YOLO
from pathlib import Path
import torch
import time

PROJECT_ROOT = Path.cwd()
RUNS_DIR = PROJECT_ROOT / "runs"
DATA_YAML_PATH = PROJECT_ROOT / "data" / "processed" / "deep_pcb_yolo" / "data.yaml"

print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("data.yaml:", DATA_YAML_PATH)

CUDA disponible: True
GPU: Tesla T4
data.yaml: /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/data.yaml


In [8]:
start_time = time.time()

model_nano = YOLO("yolo11n.pt")

results_nano = model_nano.train(
    data=str(DATA_YAML_PATH),
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name="yolo11n_deeppcb",
    workers=2,
    seed=42,
    patience=10,
    exist_ok=True
)

elapsed_time_nano = time.time() - start_time

print(f"Tiempo total YOLO11n: {elapsed_time_nano / 60:.2f} minutos")

New https://pypi.org/project/ultralytics/8.4.149 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.87 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, moment

In [9]:
for run_name in ["yolo11n_deeppcb"]:
    run_dir = RUNS_DIR / run_name
    print(f"\n{run_name}")
    print("Existe:", run_dir.exists())

    if run_dir.exists():
        for file in sorted(run_dir.iterdir()):
            print("-", file.name)


yolo11n_deeppcb
Existe: True
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- labels.jpg
- results.csv
- results.png
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
- weights


In [10]:
best_nano_path = RUNS_DIR / "yolo11n_deeppcb" / "weights" / "best.pt"

model_nano_best = YOLO(str(best_nano_path))

test_results_nano = model_nano_best.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name="yolo11n_test",
    exist_ok=True
)

Ultralytics 8.4.87 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 889.1±427.2 MB/s, size: 29.6 KB)
val: Scanning /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/labels/test... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 2.0Kit/s 0.3s
val: New cache created: /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.8it/s 8.5s
                   all        500       3140      0.703      0.724      0.789      0.484
                  open        482        659      0.852      0.627      0.805      0.433
                 short        368        478      0.662      0.402      0.531       0.29
             mousebite        413        586      0.623      0.675      0.718   

In [11]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RUNS_DIR = PROJECT_ROOT / "runs"

for path in sorted(RUNS_DIR.iterdir()):
    if path.is_dir():
        print(path)

/content/pcb-defect-detection-yolo/runs/yolo11n_deeppcb
/content/pcb-defect-detection-yolo/runs/yolo11n_test


In [12]:
import pandas as pd

run_names = [
    "yolo11n_deeppcb",
]

for run_name in run_names:
    results_path = RUNS_DIR / run_name / "results.csv"
    print(f"\n{run_name}")
    print("Existe:", results_path.exists())

    if results_path.exists():
        df = pd.read_csv(results_path)
        display(df.tail())


yolo11n_deeppcb
Existe: True


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,31.6044,2.45259,4.20708,1.52491,0.01838,0.56482,0.07018,0.01975,1.59241,4.23299,1.06674,0.000327,0.000327,0.000327
1,2,51.8239,1.70016,2.64139,1.16052,0.64902,0.06094,0.13674,0.03375,2.28265,3.75690,1.27590,0.000529,0.000529,0.000529
2,3,71.5788,1.54455,2.02458,1.09498,0.33101,0.29093,0.24422,0.06248,2.44540,2.70914,1.43910,0.000600,0.000600,0.000600
3,4,87.4307,1.41091,1.72072,1.05350,0.61421,0.60589,0.60684,0.19842,2.09479,1.89868,1.31497,0.000406,0.000406,0.000406
4,5,103.3250,1.29856,1.54527,1.02473,0.69959,0.75475,0.81554,0.50832,1.21505,1.43651,1.03638,0.000208,0.000208,0.000208


In [13]:
test_run_names = [
    "yolo11n_test",
]

for run_name in test_run_names:
    results_path = RUNS_DIR / run_name / "results.csv"
    print(f"\n{run_name}")
    print("Existe:", results_path.exists())

    if results_path.exists():
        df = pd.read_csv(results_path)
        display(df.tail())


yolo11n_test
Existe: False


In [14]:
for run_name in [
    "yolo11n_deeppcb",
    "yolo11n_test",
]:
    run_dir = RUNS_DIR / run_name

    print(f"\n{run_name}")
    if run_dir.exists():
        for file in sorted(run_dir.iterdir()):
            print("-", file.name)


yolo11n_deeppcb
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- labels.jpg
- results.csv
- results.png
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
- weights

yolo11n_test
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- confusion_matrix.png
- confusion_matrix_normalized.png
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg


In [15]:
PROJECT_ROOT = Path.cwd()
RUNS_DIR = PROJECT_ROOT / "runs"

def extract_test_metrics(results, model_name):
    """
    Extrae métricas principales de un resultado de validación YOLO.
    """
    box = results.box

    return {
        "model": model_name,
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
    }


test_metrics = pd.DataFrame([
    extract_test_metrics(test_results_nano, "YOLO11n"),
])

test_metrics

,model,precision,recall,mAP50,mAP50_95
0,YOLO11n,0.702874,0.723923,0.789238,0.483985


In [16]:
metrics_dir = PROJECT_ROOT / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

test_metrics_path = metrics_dir / "test_metrics_summary.csv"

test_metrics.to_csv(test_metrics_path, index=False)

print(f"Métricas guardadas en: {test_metrics_path}")
display(test_metrics)

Métricas guardadas en: /content/pcb-defect-detection-yolo/results/metrics/test_metrics_summary.csv


,model,precision,recall,mAP50,mAP50_95
0,YOLO11n,0.702874,0.723923,0.789238,0.483985


In [20]:
CLASS_NAMES = [
    "open",
    "short",
    "mousebite",
    "spur",
    "copper",
    "pin-hole",
]

def extract_class_metrics(results, model_name):
    """
    Extrae métricas por clase desde un resultado de validación YOLO.
    """
    box = results.box

    rows = []

    for idx, class_name in enumerate(CLASS_NAMES):
        rows.append({
            "model": model_name,
            "class_id": idx,
            "class_name": class_name,
            "precision": float(box.p[idx]),
            "recall": float(box.r[idx]),
            "mAP50": float(box.ap50[idx]),
            "mAP50_95": float(box.ap[idx]),
        })

    return rows


class_metrics = pd.DataFrame(
    extract_class_metrics(test_results_nano, "YOLO11n")
)

class_metrics_path = metrics_dir / "test_metrics_by_class.csv"

class_metrics.to_csv(class_metrics_path, index=False)

display(class_metrics)
print(f"Métricas por clase guardadas en: {class_metrics_path}")

,model,class_id,class_name,precision,recall,mAP50,mAP50_95
0,YOLO11n,0,open,0.851520,0.626575,0.805170,0.432832
1,YOLO11n,1,short,0.662157,0.401832,0.530687,0.289569
2,YOLO11n,2,mousebite,0.623253,0.674706,0.717889,0.410127
3,YOLO11n,3,spur,0.789049,0.704720,0.807349,0.460976
4,YOLO11n,4,copper,0.703720,0.963362,0.951984,0.678342
5,YOLO11n,5,pin-hole,0.587545,0.972340,0.922351,0.632063


Métricas por clase guardadas en: /content/pcb-defect-detection-yolo/results/metrics/test_metrics_by_class.csv


In [21]:
from pathlib import Path
from datetime import datetime
import shutil
import json
import joblib
import ultralytics
import torch


# ============================================================
# REGISTRO Y VERSIONADO DEL MODELO
# ============================================================

PROJECT_ROOT = Path.cwd()

MODEL_REGISTRY_DIR = PROJECT_ROOT / "models" / "registry"
MODEL_REGISTRY_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. GENERAR IDENTIFICADOR DE VERSIÓN
# ------------------------------------------------------------

timestamp = datetime.now()

version = timestamp.strftime("%Y%m%d_%H%M%S")
model_name = "yolo11n_deeppcb"

version_name = f"{model_name}_{version}"

version_dir = MODEL_REGISTRY_DIR / version_name
version_dir.mkdir(parents=True, exist_ok=False)

print(f"Registrando versión: {version_name}")
print(f"Directorio: {version_dir}")

Registrando versión: yolo11n_deeppcb_20260912_135338
Directorio: /content/pcb-defect-detection-yolo/models/registry/yolo11n_deeppcb_20260912_135338


In [22]:
# ============================================================
# 2. GUARDAR MODELO .PT
# ============================================================

source_pt = RUNS_DIR / "yolo11n_deeppcb" / "weights" / "best.pt"

pt_path = version_dir / "model.pt"

shutil.copy2(
    source_pt,
    pt_path
)

print(f"Modelo .pt guardado en:")
print(pt_path)

Modelo .pt guardado en:
/content/pcb-defect-detection-yolo/models/registry/yolo11n_deeppcb_20260912_135338/model.pt


In [23]:
# ============================================================
# 3. PREPARAR MÉTRICAS Y METADATOS
# ============================================================

global_metrics = {
    "precision": float(test_metrics.iloc[0]["precision"]),
    "recall": float(test_metrics.iloc[0]["recall"]),
    "mAP50": float(test_metrics.iloc[0]["mAP50"]),
    "mAP50_95": float(test_metrics.iloc[0]["mAP50_95"]),
}


class_metrics_dict = (
    class_metrics
    .drop(columns=["model"])
    .to_dict(orient="records")
)


metadata = {
    "model_name": model_name,
    "version": version,
    "registered_at": timestamp.isoformat(),

    "architecture": "YOLO11n",
    "task": "object_detection",
    "dataset": "DeepPCB",

    "num_classes": len(CLASS_NAMES),
    "class_names": CLASS_NAMES,

    "training": {
        "epochs": 5,
        "image_size": 640,
        "batch_size": 16,
        "seed": 42,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "training_time_minutes": round(elapsed_time_nano / 60, 2),
    },

    "metrics": global_metrics,

    "metrics_by_class": class_metrics_dict,

    "environment": {
        "torch_version": torch.__version__,
        "ultralytics_version": ultralytics.__version__,
    },

    "artifacts": {
        "pytorch_model": "model.pt",
        "joblib_model": "model.joblib",
        "metadata": "metadata.json",
    },
}

In [24]:
# ============================================================
# 4. EXPORTAR METADATOS A JSON
# ============================================================

json_path = version_dir / "metadata.json"

with open(
    json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


print(f"Metadata guardada en:")
print(json_path)

Metadata guardada en:
/content/pcb-defect-detection-yolo/models/registry/yolo11n_deeppcb_20260912_135338/metadata.json


In [25]:
# ============================================================
# 5. EXPORTAR MODELO A JOBLIB
# ============================================================

joblib_path = version_dir / "model.joblib"

model_package = {
    "model": model_nano_best,
    "metadata": metadata,
}

joblib.dump(
    model_package,
    joblib_path
)

print(f"Modelo .joblib guardado en:")
print(joblib_path)

Modelo .joblib guardado en:
/content/pcb-defect-detection-yolo/models/registry/yolo11n_deeppcb_20260912_135338/model.joblib


In [26]:
# ============================================================
# 6. VERIFICAR REGISTRO
# ============================================================

print("=" * 70)
print("MODELO REGISTRADO CORRECTAMENTE")
print("=" * 70)

print(f"\nModelo:   {model_name}")
print(f"Versión:  {version}")
print(f"Fecha:    {timestamp}")

print("\nMétricas:")
print(f"Precision: {global_metrics['precision']:.4f}")
print(f"Recall:    {global_metrics['recall']:.4f}")
print(f"mAP@50:    {global_metrics['mAP50']:.4f}")
print(f"mAP@50-95: {global_metrics['mAP50_95']:.4f}")

print("\nArchivos generados:")

for file in sorted(version_dir.iterdir()):
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"- {file.name}: {size_mb:.2f} MB")

MODELO REGISTRADO CORRECTAMENTE

Modelo:   yolo11n_deeppcb
Versión:  20260912_135338
Fecha:    2026-09-12 13:53:38.133263

Métricas:
Precision: 0.7029
Recall:    0.7239
mAP@50:    0.7892
mAP@50-95: 0.4840

Archivos generados:
- metadata.json: 0.00 MB
- model.joblib: 10.29 MB
- model.pt: 5.22 MB
